# hybraut watchdog

The runtime system is monitored and managed by a **Watchdog Finite State Machine (FSM)**. This FSM listens to runtime events and responds accordingly to maintain mission continuity, recover from failures, and safely shut down in critical conditions. It helps enforce lifecycle safety and determinism across automaton operations.

When specific events are published (e.g., mode transitions, errors, mission completion), the FSM interprets these and updates the automaton's state. Each FSM state is also associated with a corresponding **status message**, which is published to inform external systems about the automaton's condition.

![hybraut_watchdog_fsm_png](.github/diagrams/hybraut_watchdog_fsm_flowchart.png)

In [12]:
""" 
a simple script for testing the hybraut watchdog FSMp
"""

import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, qos_profile_system_default
from rclpy.executors import MultiThreadedExecutor, Executor
import threading
from hybraut_interfaces.msg import AutomatonEvents
import os
# import sys

# # sys.path.append(os.path.dirname(__file__))

# from hybraut_watchdog import HybrautWatchdogFSM
from hybraut_executor.watchdog import HybrautWatchdogFSM

def main():
    rclpy.init()
    executor: Executor = MultiThreadedExecutor(num_threads=os.cpu_count())
    node = Node('mock_node')
    executor.add_node(node)
    
    thread = threading.Thread(target=executor.spin, daemon=True)
    thread.start()
    
    try:
        watchdogFSM: HybrautWatchdogFSM = HybrautWatchdogFSM(node)
        # Test 1. transition to TRANSITIONING STATE
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.TRANSITION_GUARD_ENABLED
        ))
        #Test 2: Transition back to active state
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.TRANSITION_COMPLETE
        ))
        # Test 3: Transition to error state
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.RECOVERABLE_ERROR
        ))
        # Test 4: Attempt fix
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.ATTEMPT_FIX
        ))
        # Test 5: recovered
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.RECOVERED
        ))
        # # TODO: FIX Test 6: transition and recoverable error 
        # watchdogFSM.trigger_transition(AutomatonEvents(
        #     type=AutomatonEvents.TRANSITION_GUARD_ENABLED
        # ))
        # watchdogFSM.trigger_transition(AutomatonEvents(
        #     type=AutomatonEvents.RECOVERABLE_ERROR # TODO: need to fix this transition it is not being recognised
        # ))
        # Test 7: fatal 
        
        # Test 8: Mission Complete
        watchdogFSM.trigger_transition(AutomatonEvents(
            type=AutomatonEvents.MISSION_COMPLETE
        ))
    except KeyboardInterrupt:
        print ("Shutting down gracefully...")
    finally:
        executor.shutdown()
        thread.join()
        node.destroy_node()
        
    rclpy.shutdown()

if __name__ == '__main__':
    main()

[INFO] [1753817877.553636061] [mock_node]: Publishing status: StatusEnum.TRANSITIONING
[INFO] [1753817877.574541391] [mock_node]: transition completed, current_state: StatusEnum.TRANSITIONING
[INFO] [1753817877.575698016] [mock_node]: Publishing status: StatusEnum.ACTIVE
[INFO] [1753817877.596728690] [mock_node]: transition completed, current_state: StatusEnum.ACTIVE
[INFO] [1753817877.597686642] [mock_node]: Publishing status: StatusEnum.ERROR
[INFO] [1753817877.618619208] [mock_node]: transition completed, current_state: StatusEnum.ERROR
[INFO] [1753817877.620348522] [mock_node]: Publishing status: StatusEnum.RECOVERING
[INFO] [1753817877.641308397] [mock_node]: transition completed, current_state: StatusEnum.RECOVERING
[INFO] [1753817877.642222247] [mock_node]: Publishing status: StatusEnum.ACTIVE
[INFO] [1753817877.663904043] [mock_node]: transition completed, current_state: StatusEnum.ACTIVE
[INFO] [1753817877.664940564] [mock_node]: Publishing status: StatusEnum.MISSION_COMPLETE
